
# Домашнее задание по предмету «Прикладные задачи анализа данных»

Этот ноутбук полностью покрывает требования из задания: подготовка окружения, классификация Rotten Tomatoes, эксперименты с заморозкой слоёв, few-shot с SetFit, дообучение MLM и NER на CoNLL-2003. Требования взяты из приложенного PDF-файла. fileciteturn0file0L1-L63

Все пояснения и комментарии даны на русском языке. Ноутбук рассчитан на последовательный запуск сверху вниз.



## Задание 1. Подготовка окружения, импортов и воспроизводимости

Ниже задаются импорты, фиксируется seed и определяется устройство. Это нужно для воспроизводимости экспериментов и стабильности результатов при обучении трансформеров. fileciteturn0file0L10-L18


In [1]:
print("lalala")

lalala


In [2]:
pip install datasets

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached xxhash-3.6.0-cp311-cp311-win_amd64.whl.metadata (13 kB)
     ---------------------------------------- 0.0/41.4 kB ? eta -:--:--
     ---------------------------------------- 41.4/41.4 kB 1.9 MB/s eta 0:00:00
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp311-cp311-win_amd64.whl.metadata (21 kB)
  Using cached propcache-0.4.1-cp311-cp311-win_amd64.whl.metadata (14 kB)
     ---------------------------------------- 0.0/82.2 kB ? eta -:--:--
     ---------------------------------------- 82.2/82.2 kB 2.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------- ----------------------------- 143.4/527.0 kB 2.8 MB/s eta 0:00:01
   --------------------------- ------------ 368.6/527.0 kB 4.6 MB/s eta 0:00:01
   -------------------------------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install evaluate

   ---------------------------------------- 0.0/84.1 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/84.1 kB ? eta -:--:--
   ------------------- -------------------- 41.0/84.1 kB 495.5 kB/s eta 0:00:01
   ---------------------------------------- 84.1/84.1 kB 785.4 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:

# Опционально: раскомментируйте, если среда не подготовлена.
# !pip -q install -U transformers datasets evaluate accelerate scikit-learn setfit sentence-transformers seqeval

import os
import gc
import time
import math
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import evaluate

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    AutoModelForTokenClassification,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    pipeline,
)

warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Выбранное устройство: {DEVICE}')
if DEVICE == 'cuda':
    print('Название GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU недоступен, вычисления будут выполняться на CPU.')

print(f'SEED зафиксирован: {SEED}')
print('Импорты выполнены успешно.')
print('Версии библиотек:')
print('  transformers =', transformers.__version__)
print('  datasets     =', datasets.__version__)
print('  torch        =', torch.__version__)


c:\Users\anmrt\Desktop\Useful shit\nlp_labs_8sem\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Выбранное устройство: cpu
GPU недоступен, вычисления будут выполняться на CPU.
SEED зафиксирован: 42
Импорты выполнены успешно.
Версии библиотек:
  transformers = 5.1.0
  datasets     = 4.8.4
  torch        = 2.10.0+cpu


In [5]:

BASE_MODEL_NAME = 'bert-base-cased'
CLS_OUTPUT_ROOT = Path('outputs_cls')
MLM_OUTPUT_DIR = Path('mlm')
NER_OUTPUT_DIR = Path('ner_model')

CLS_OUTPUT_ROOT.mkdir(exist_ok=True)
MLM_OUTPUT_DIR.mkdir(exist_ok=True)
NER_OUTPUT_DIR.mkdir(exist_ok=True)

print('Рабочие папки созданы:', CLS_OUTPUT_ROOT, MLM_OUTPUT_DIR, NER_OUTPUT_DIR)
print('Короткий вывод: теперь ноутбук работает в согласованном окружении, а результаты экспериментов будут сохраняться в отдельные директории.')


Рабочие папки созданы: outputs_cls mlm ner_model
Короткий вывод: теперь ноутбук работает в согласованном окружении, а результаты экспериментов будут сохраняться в отдельные директории.



## Задание 2. Загрузка и первичный анализ данных для классификации

Загружаем датасет Rotten Tomatoes, изучаем размеры сплитов, примеры и баланс классов. Это нужно, чтобы понимать постановку задачи и корректно интерпретировать метрики классификации. fileciteturn0file0L19-L31


In [6]:

rt_ds = load_dataset('rotten_tomatoes')
train_ds = rt_ds['train']
test_ds = rt_ds['test']

print(rt_ds)
print(f'Размер train: {len(train_ds)}')
print(f'Размер test:  {len(test_ds)}')

print('Пример 1:')
print(train_ds[0])
print('Пример 2:')
print(train_ds[1])

label_names = train_ds.features['label'].names
train_label_counts = Counter(train_ds['label'])
test_label_counts = Counter(test_ds['label'])

label_dist_df = pd.DataFrame({
    'label_id': list(range(len(label_names))),
    'label_name': label_names,
    'train_count': [train_label_counts[i] for i in range(len(label_names))],
    'test_count': [test_label_counts[i] for i in range(len(label_names))],
})
label_dist_df


Generating test split: 100%|██████████| 1066/1066 [00:00<00:00, 284133.71 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})
Размер train: 8530
Размер test:  1066
Пример 1:
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}
Пример 2:
{'text': 'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-earth .', 'label': 1}


,label_id,label_name,train_count,test_count
0,0,neg,4265,533
1,1,pos,4265,533


In [7]:

print('Пояснение к задаче: мы классифицируем тональность отзыва о фильме.')
for idx, name in enumerate(label_names):
    print(f'  Метка {idx} -> {name}')

train_share = label_dist_df['train_count'] / label_dist_df['train_count'].sum()
max_imbalance = float((train_share.max() - train_share.min()) * 100)

print('Короткая интерпретация результата:')
print('Датасет содержит тексты отзывов и бинарные метки тональности.')
print('По распределению классов видно, насколько задача сбалансирована на обучающей и тестовой выборках.')
if max_imbalance < 5:
    print('Классы распределены довольно ровно, поэтому accuracy и F1 можно интерпретировать без сильной поправки на дисбаланс.')
else:
    print('Есть заметный дисбаланс классов, поэтому кроме accuracy особенно важно смотреть на F1.')


Пояснение к задаче: мы классифицируем тональность отзыва о фильме.
  Метка 0 -> neg
  Метка 1 -> pos
Короткая интерпретация результата:
Датасет содержит тексты отзывов и бинарные метки тональности.
По распределению классов видно, насколько задача сбалансирована на обучающей и тестовой выборках.
Классы распределены довольно ровно, поэтому accuracy и F1 можно интерпретировать без сильной поправки на дисбаланс.



## Задание 3. Токенизация и подготовка батчей

Преобразуем текст в токены BERT и используем динамический padding внутри батча через `DataCollatorWithPadding`. Это позволяет не паддить весь датасет до одной глобальной длины и экономить память. fileciteturn0file0L32-L41


In [8]:

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)


def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True)


tokenized_rt = rt_ds.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print('Структура токенизированной записи:')
example_tok = tokenized_rt['train'][0]
for key, value in example_tok.items():
    if isinstance(value, list):
        print(f'{key}: {value[:20]} ... (длина={len(value)})')
    else:
        print(f'{key}: {value}')


Map: 100%|██████████| 1066/1066 [00:00<00:00, 15964.10 examples/s]

Структура токенизированной записи:
text: the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
label: 1
input_ids: [101, 1103, 2067, 1110, 17348, 1106, 1129, 1103, 6880, 1432, 112, 188, 1207, 107, 14255, 1389, 107, 1105, 1115, 1119] ... (длина=57)
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] ... (длина=57)
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] ... (длина=57)


In [9]:

print('Короткая интерпретация результата:')
print('Параметр truncation=True нужен, чтобы слишком длинные тексты не выходили за максимальную длину модели и не вызывали ошибок по размерности.')
print('Динамический padding добавляет паддинг только до максимальной длины внутри текущего батча, а не по всему датасету.')
print('Это уменьшает расход памяти и обычно ускоряет обучение по сравнению с глобальным padding до фиксированного максимума.')


Короткая интерпретация результата:
Параметр truncation=True нужен, чтобы слишком длинные тексты не выходили за максимальную длину модели и не вызывали ошибок по размерности.
Динамический padding добавляет паддинг только до максимальной длины внутри текущего батча, а не по всему датасету.
Это уменьшает расход памяти и обычно ускоряет обучение по сравнению с глобальным padding до фиксированного максимума.



## Вспомогательные функции для экспериментов классификации


In [10]:

acc_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')


def compute_cls_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = acc_metric.compute(predictions=preds, references=labels)['accuracy']
    f1 = f1_metric.compute(predictions=preds, references=labels, average='binary')['f1']
    return {'accuracy': accuracy, 'f1': f1}


def count_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def human_readable_params(n):
    if n >= 1_000_000:
        return f'{n/1_000_000:.2f} млн'
    if n >= 1_000:
        return f'{n/1_000:.2f} тыс.'
    return str(n)


def freeze_all_bert_body(model):
    for param in model.bert.embeddings.parameters():
        param.requires_grad = False
    for param in model.bert.encoder.parameters():
        param.requires_grad = False
    return model


def freeze_first_n_bert_layers(model, n_freeze):
    for param in model.bert.embeddings.parameters():
        param.requires_grad = False if n_freeze > 0 else True
    for layer_idx, layer in enumerate(model.bert.encoder.layer):
        requires_grad = not (layer_idx < n_freeze)
        for param in layer.parameters():
            param.requires_grad = requires_grad
    return model


def build_cls_model(num_labels=2):
    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME, num_labels=num_labels)
    return model


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def train_and_evaluate_sequence_model(
    experiment_name,
    freeze_mode='none',
    n_freeze=None,
    num_train_epochs=1,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
):
    cleanup()
    model = build_cls_model(num_labels=2)
    total_params_before, trainable_before = count_trainable_params(model)

    if freeze_mode == 'all_body':
        model = freeze_all_bert_body(model)
    elif freeze_mode == 'partial':
        model = freeze_first_n_bert_layers(model, n_freeze=n_freeze)

    total_params_after, trainable_after = count_trainable_params(model)

    output_dir = CLS_OUTPUT_ROOT / experiment_name
    output_dir.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        overwrite_output_dir=True,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='steps',
        logging_steps=100,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        weight_decay=weight_decay,
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        save_total_limit=1,
        report_to=[],
        seed=SEED,
        dataloader_num_workers=0,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_rt['train'],
        eval_dataset=tokenized_rt['test'],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_cls_metrics,
    )

    start_time = time.time()
    trainer.train()
    elapsed = time.time() - start_time
    eval_metrics = trainer.evaluate(tokenized_rt['test'])

    result = {
        'experiment': experiment_name,
        'freeze_mode': freeze_mode,
        'n_freeze': n_freeze if n_freeze is not None else 0,
        'total_params': total_params_after,
        'trainable_params_before': trainable_before,
        'trainable_params_after': trainable_after,
        'trainable_share': trainable_after / total_params_after,
        'accuracy': eval_metrics.get('eval_accuracy'),
        'f1': eval_metrics.get('eval_f1'),
        'eval_loss': eval_metrics.get('eval_loss'),
        'runtime_sec': elapsed,
        'output_dir': str(output_dir),
    }

    cleanup()
    return result, trainer



## Задание 4. Базовый fine-tuning для классификации через Trainer

Это базовый контрольный эксперимент: обучаем `bert-base-cased` для бинарной классификации и оцениваем качество на тесте. fileciteturn0file0L42-L48


In [13]:
pip install -U transformers

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: HTTPSConnectionPool(host='files.pythonhosted.org', port=443): Max retries exceeded with url: /packages/b8/88/ae8320064e32679a5429a2c9ebbc05c2bf32cefb6e076f9b07f6d685a9b4/transformers-5.3.0-py3-none-any.whl.metadata (Caused by ReadTimeoutError("HTTPSConnectionPool(host='files.pythonhosted.org', port=443): Read timed out. (read timeout=15)"))


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
baseline_result, baseline_trainer = train_and_evaluate_sequence_model(
    experiment_name='baseline_full_finetune',
    freeze_mode='none',
    num_train_epochs=1,
    learning_rate=2e-5,
)

pd.DataFrame([baseline_result])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 864.75it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those para

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'overwrite_output_dir'

In [ ]:

print('Итоговые метрики baseline на test:')
print(f"accuracy = {baseline_result['accuracy']:.4f}")
print(f"f1       = {baseline_result['f1']:.4f}")
print('Короткая интерпретация результата:')
if baseline_result['f1'] >= 0.88:
    print('Качество получилось высоким: модель хорошо адаптировалась к задаче бинарной классификации тональности.')
elif baseline_result['f1'] >= 0.80:
    print('Качество получилось хорошим, но остаётся пространство для ошибок на неоднозначных или ироничных отзывах.')
else:
    print('Качество умеренное: вероятно, модели не хватило эпох, ресурсов или более тонкой настройки гиперпараметров.')
print('Ошибки обычно возникают на коротких, саркастических или контекстно неоднозначных фразах, где тональность выражена неявно.')



## Задание 5. Эксперимент: полная заморозка BERT

Оставляем обучаемой только классификационную голову и сравниваем этот режим с полным fine-tuning. fileciteturn0file0L49-L57


In [ ]:

frozen_all_result, frozen_all_trainer = train_and_evaluate_sequence_model(
    experiment_name='frozen_all_body',
    freeze_mode='all_body',
    num_train_epochs=1,
    learning_rate=2e-4,
)

comparison_2 = pd.DataFrame([
    {
        'experiment': baseline_result['experiment'],
        'trainable_params': baseline_result['trainable_params_after'],
        'accuracy': baseline_result['accuracy'],
        'f1': baseline_result['f1'],
    },
    {
        'experiment': frozen_all_result['experiment'],
        'trainable_params': frozen_all_result['trainable_params_after'],
        'accuracy': frozen_all_result['accuracy'],
        'f1': frozen_all_result['f1'],
    },
])
comparison_2


In [ ]:

print('Число обучаемых параметров до заморозки:', human_readable_params(frozen_all_result['trainable_params_before']))
print('Число обучаемых параметров после заморозки:', human_readable_params(frozen_all_result['trainable_params_after']))

print('Короткий вывод:')
print('Полная заморозка резко уменьшает число обучаемых параметров и делает обучение заметно дешевле.')
if frozen_all_result['f1'] < baseline_result['f1']:
    print('При этом качество ожидаемо ниже baseline, потому что представления BERT не подстраиваются под конкретную задачу тональности.')
else:
    print('Интересно, что качество почти не просело: это означает, что предобученные признаки BERT уже достаточно хороши для данной задачи.')
print('Такой режим полезен, когда важнее скорость и экономия памяти, чем максимальное качество.')



## Задание 6. Эксперимент: заморозка первых 5 блоков encoder

Замораживаем только нижние слои BERT, а верхние оставляем обучаемыми. Это компромисс между качеством и вычислительной стоимостью. fileciteturn0file0L58-L63


In [ ]:

freeze_5_result, freeze_5_trainer = train_and_evaluate_sequence_model(
    experiment_name='freeze_first_5_layers',
    freeze_mode='partial',
    n_freeze=5,
    num_train_epochs=1,
    learning_rate=2e-5,
)

comparison_3 = pd.DataFrame([
    {
        'experiment': baseline_result['experiment'],
        'trainable_params': baseline_result['trainable_params_after'],
        'accuracy': baseline_result['accuracy'],
        'f1': baseline_result['f1'],
    },
    {
        'experiment': frozen_all_result['experiment'],
        'trainable_params': frozen_all_result['trainable_params_after'],
        'accuracy': frozen_all_result['accuracy'],
        'f1': frozen_all_result['f1'],
    },
    {
        'experiment': freeze_5_result['experiment'],
        'trainable_params': freeze_5_result['trainable_params_after'],
        'accuracy': freeze_5_result['accuracy'],
        'f1': freeze_5_result['f1'],
    },
])
comparison_3


In [ ]:

print('Обучаемые параметры в режиме freeze-1-5:', human_readable_params(freeze_5_result['trainable_params_after']))
print(f"accuracy = {freeze_5_result['accuracy']:.4f}")
print(f"f1       = {freeze_5_result['f1']:.4f}")

print('Короткая интерпретация результата:')
print('Частичная заморозка сохраняет возможность адаптировать верхние слои под задачу, поэтому качество обычно ближе к baseline, чем при полной заморозке.')
if freeze_5_result['f1'] >= frozen_all_result['f1']:
    print('Видно, что адаптация верхних блоков действительно помогает модели лучше улавливать задачу тональности.')
print('Нижние слои чаще кодируют более общие языковые признаки, поэтому их фиксация не всегда сильно вредит качеству.')



## Задание 7. Универсальная функция заморозки N блоков и мини-sweep

Проверяем несколько уровней заморозки и собираем итоговую таблицу качества и числа обучаемых параметров. Это уже небольшой исследовательский этап. fileciteturn0file0L64-L72


In [ ]:


def freeze_bert_layers(model, n_freeze):
    return freeze_first_n_bert_layers(model, n_freeze=n_freeze)


sweep_results = []
for n_freeze in [0, 3, 6, 9, 12]:
    result, _ = train_and_evaluate_sequence_model(
        experiment_name=f'sweep_freeze_{n_freeze}',
        freeze_mode='partial' if n_freeze > 0 else 'none',
        n_freeze=n_freeze if n_freeze > 0 else 0,
        num_train_epochs=1,
        learning_rate=2e-5 if n_freeze < 12 else 2e-4,
    )
    sweep_results.append(result)

sweep_df = pd.DataFrame(sweep_results)[['n_freeze', 'trainable_params_after', 'trainable_share', 'accuracy', 'f1', 'runtime_sec']]
sweep_df = sweep_df.sort_values('n_freeze').reset_index(drop=True)
sweep_df


In [ ]:

best_sweep_row = sweep_df.sort_values(['f1', 'accuracy'], ascending=False).iloc[0]
print('Итоговая таблица sweep показана выше.')
print('Краткий вывод:')
print(f"Лучшая конфигурация по F1 в этом запуске: n_freeze = {int(best_sweep_row['n_freeze'])}.")
print('При увеличении числа замороженных блоков число обучаемых параметров уменьшается, но вместе с этим модель постепенно теряет гибкость адаптации.')
print('Оптимальной обычно выглядит конфигурация, где качество ещё близко к baseline, а число обучаемых параметров уже заметно снижено.')
print('Такой sweet spot полезен, когда нужно ускорить обучение без слишком большого падения качества.')



## Задание 8. Few-shot классификация с SetFit

Формируем небольшой few-shot train и сравниваем SetFit с полноценным fine-tuning BERT. fileciteturn0file0L73-L80


In [ ]:

# Формируем few-shot train: по 16 примеров на класс.
shots_per_class = 16
few_shot_parts = []
for label_id in sorted(set(train_ds['label'])):
    subset = train_ds.filter(lambda x: x['label'] == label_id)
    subset = subset.shuffle(seed=SEED).select(range(shots_per_class))
    few_shot_parts.append(subset)

few_shot_train = datasets.concatenate_datasets(few_shot_parts).shuffle(seed=SEED)
print(f'Размер few-shot train: {len(few_shot_train)}')
print(few_shot_train[:3])


In [ ]:

try:
    from setfit import SetFitModel, Trainer as SetFitTrainer, TrainingArguments as SetFitTrainingArguments
except ImportError:
    from setfit import SetFitModel, SetFitTrainer
    SetFitTrainingArguments = None

setfit_model = SetFitModel.from_pretrained('sentence-transformers/paraphrase-mpnet-base-v2')

start_time = time.time()
if SetFitTrainingArguments is not None:
    setfit_args = SetFitTrainingArguments(
        output_dir='setfit_output',
        batch_size=16,
        num_epochs=1,
        num_iterations=20,
        seed=SEED,
        report_to='none',
    )
    setfit_trainer = SetFitTrainer(
        model=setfit_model,
        args=setfit_args,
        train_dataset=few_shot_train,
        eval_dataset=test_ds,
        column_mapping={'text': 'text', 'label': 'label'},
    )
else:
    setfit_trainer = SetFitTrainer(
        model=setfit_model,
        train_dataset=few_shot_train,
        eval_dataset=test_ds,
        column_mapping={'text': 'text', 'label': 'label'},
        batch_size=16,
        num_epochs=1,
        num_iterations=20,
        seed=SEED,
    )

setfit_trainer.train()
setfit_pred = setfit_model(few_shot_train['text'][:2])
setfit_test_preds = setfit_model(test_ds['text'])
setfit_acc = acc_metric.compute(predictions=setfit_test_preds, references=test_ds['label'])['accuracy']
setfit_f1 = f1_metric.compute(predictions=setfit_test_preds, references=test_ds['label'], average='binary')['f1']
setfit_runtime = time.time() - start_time

setfit_result = {
    'experiment': 'setfit_few_shot_16_per_class',
    'shots_per_class': shots_per_class,
    'train_size': len(few_shot_train),
    'accuracy': setfit_acc,
    'f1': setfit_f1,
    'runtime_sec': setfit_runtime,
}

pd.DataFrame([setfit_result])


In [ ]:

print(f"Few-shot SetFit: accuracy = {setfit_result['accuracy']:.4f}, f1 = {setfit_result['f1']:.4f}")
print('Краткое сравнение с baseline:')
print(f"Baseline F1 = {baseline_result['f1']:.4f}")
print(f"SetFit   F1 = {setfit_result['f1']:.4f}")
print('Короткая интерпретация результата:')
print('SetFit полезен в режиме малого числа размеченных примеров, когда полный fine-tuning трансформера может быть избыточным или слишком дорогим.')
if setfit_result['f1'] >= baseline_result['f1']:
    print('В этом запуске SetFit оказался очень конкурентоспособным, что показывает силу embedding-подхода в few-shot сценарии.')
else:
    print('В этом запуске baseline лучше, что ожидаемо: у полного fine-tuning больше данных и выше адаптивность модели.')
print('SetFit особенно полезен, когда разметки мало, а получить быстрый и устойчивый результат важнее, чем выжать максимум качества.')



## Задание 9. Дообучение под MLM

Дообучаем `bert-base-cased` как masked language model на текстах Rotten Tomatoes, сохраняем модель в папку `mlm/` и сравниваем `fill-mask` до и после дообучения. fileciteturn0file0L81-L90


In [ ]:

mlm_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)


def preprocess_mlm(examples):
    return mlm_tokenizer(
        examples['text'],
        truncation=True,
        padding=False,
        return_special_tokens_mask=True,
        max_length=128,
    )


mlm_ds = rt_ds.map(preprocess_mlm, batched=True, remove_columns=['text', 'label'])
mlm_collator = DataCollatorForLanguageModeling(tokenizer=mlm_tokenizer, mlm_probability=0.15)
mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL_NAME)

mlm_args = TrainingArguments(
    output_dir='mlm_output',
    overwrite_output_dir=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=100,
    num_train_epochs=1,
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=1,
    report_to=[],
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=mlm_ds['train'],
    eval_dataset=mlm_ds['test'],
    data_collator=mlm_collator,
    tokenizer=mlm_tokenizer,
)

mlm_trainer.train()
mlm_eval = mlm_trainer.evaluate(mlm_ds['test'])
mlm_trainer.save_model(MLM_OUTPUT_DIR)
mlm_tokenizer.save_pretrained(MLM_OUTPUT_DIR)

print('MLM-модель и токенайзер сохранены в:', MLM_OUTPUT_DIR.resolve())
print('MLM eval_loss:', mlm_eval['eval_loss'])


In [ ]:

base_fill_mask = pipeline('fill-mask', model=BASE_MODEL_NAME, tokenizer=BASE_MODEL_NAME, device=0 if torch.cuda.is_available() else -1)
tuned_fill_mask = pipeline('fill-mask', model=str(MLM_OUTPUT_DIR), tokenizer=str(MLM_OUTPUT_DIR), device=0 if torch.cuda.is_available() else -1)

mask_examples = [
    'The movie was absolutely [MASK].',
    'This film is one of the most [MASK] dramas of the year.',
    'The acting was [MASK], but the script was weak.',
]


def top_words(pipe, text, top_k=5):
    preds = pipe(text, top_k=top_k)
    return [p['token_str'].strip() for p in preds]


mlm_compare_rows = []
for text in mask_examples:
    mlm_compare_rows.append({
        'text_with_mask': text,
        'base_top5': ', '.join(top_words(base_fill_mask, text, top_k=5)),
        'tuned_top5': ', '.join(top_words(tuned_fill_mask, text, top_k=5)),
    })

mlm_compare_df = pd.DataFrame(mlm_compare_rows)
mlm_compare_df


In [ ]:

print('Краткий вывод по MLM:')
print('После дообучения модель должна чаще предлагать слова, характерные для домена киноотзывов и эмоциональных оценок фильмов.')
print('Если после MLM среди предсказаний чаще появляются слова вроде moving, boring, brilliant, awful, touching и похожие, это похоже на доменную адаптацию.')
print('Такое дообучение помогает приблизить языковые представления модели к специфике текстов, на которых затем решается прикладная задача.')



## Задание 10. NER: fine-tuning на CoNLL-2003 с выравниванием меток

Загружаем CoNLL-2003, реализуем выравнивание меток под подсловную токенизацию и обучаем модель для распознавания именованных сущностей. Модель сохраняется в папку `ner_model/`. fileciteturn0file0L91-L106


In [ ]:

conll = load_dataset('conll2003')
ner_label_names = conll['train'].features['ner_tags'].feature.names
label2id = {label: i for i, label in enumerate(ner_label_names)}
id2label = {i: label for label, i in label2id.items()}

print('NER-метки:')
print(ner_label_names)


In [ ]:

ner_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
seqeval_metric = evaluate.load('seqeval')


def tokenize_and_align_labels(examples):
    tokenized_inputs = ner_tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
    )

    labels = []
    for i, word_labels in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(word_labels[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs['labels'] = labels
    return tokenized_inputs


tokenized_conll = conll.map(tokenize_and_align_labels, batched=True)



def compute_ner_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for prediction, label in zip(predictions, labels):
        current_preds = []
        current_labels = []
        for p, l in zip(prediction, label):
            if l != -100:
                current_preds.append(ner_label_names[p])
                current_labels.append(ner_label_names[l])
        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy'],
    }


In [ ]:

ner_model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=len(ner_label_names),
    id2label=id2label,
    label2id=label2id,
)

ner_args = TrainingArguments(
    output_dir='ner_output',
    overwrite_output_dir=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=100,
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=1,
    report_to=[],
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

ner_trainer = Trainer(
    model=ner_model,
    args=ner_args,
    train_dataset=tokenized_conll['train'],
    eval_dataset=tokenized_conll['validation'],
    tokenizer=ner_tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=ner_tokenizer),
    compute_metrics=compute_ner_metrics,
)

ner_trainer.train()
ner_eval = ner_trainer.evaluate(tokenized_conll['test'])
ner_trainer.save_model(NER_OUTPUT_DIR)
ner_tokenizer.save_pretrained(NER_OUTPUT_DIR)

print('NER-модель сохранена в:', NER_OUTPUT_DIR.resolve())
pd.DataFrame([ner_eval])


In [ ]:

ner_pipe = pipeline(
    'ner',
    model=str(NER_OUTPUT_DIR),
    tokenizer=str(NER_OUTPUT_DIR),
    aggregation_strategy='simple',
    device=0 if torch.cuda.is_available() else -1,
)

sample_texts = [
    'Barack Obama visited Berlin in 2013 and met Angela Merkel.',
    'Apple is opening a new office in London next year.',
]

for text in sample_texts:
    print('=' * 100)
    print('Текст:', text)
    print('Извлечённые сущности:')
    for ent in ner_pipe(text):
        print(ent)


In [ ]:

print('Короткая интерпретация результата:')
print('Значение -100 используется для игнорирования спецтокенов и всех подпоследовательных под-токенов слова при вычислении функции потерь.')
print('Это нужно, чтобы модель не наказывалась несколько раз за одно и то же исходное слово после wordpiece-разбиения.')
print('Без корректного align метки будут сдвигаться относительно токенов, а обучение станет некорректным и метрики NER ухудшатся.')
print('Именно поэтому в задачах токен-классификации выравнивание меток является обязательным этапом подготовки данных.')



## Задание 11. Итоговая сводка и выводы

Собираем результаты в общую таблицу и формулируем выводы по качеству, ресурсам, применимости SetFit, пользе MLM и итогам NER. fileciteturn0file0L107-L123


In [ ]:

summary_rows = [
    {
        'Подход': 'Baseline классификация',
        'Accuracy / Precision': baseline_result['accuracy'],
        'F1 / Recall': baseline_result['f1'],
        'Пояснение': 'Полный fine-tuning bert-base-cased на Rotten Tomatoes',
    },
    {
        'Подход': 'Frozen-head-only',
        'Accuracy / Precision': frozen_all_result['accuracy'],
        'F1 / Recall': frozen_all_result['f1'],
        'Пояснение': 'Заморожены embeddings и encoder, обучается только голова',
    },
    {
        'Подход': 'Freeze-1-5',
        'Accuracy / Precision': freeze_5_result['accuracy'],
        'F1 / Recall': freeze_5_result['f1'],
        'Пояснение': 'Заморожены первые 5 блоков encoder',
    },
    {
        'Подход': 'Few-shot SetFit',
        'Accuracy / Precision': setfit_result['accuracy'],
        'F1 / Recall': setfit_result['f1'],
        'Пояснение': f"Few-shot обучение: {shots_per_class} примеров на класс",
    },
    {
        'Подход': 'MLM',
        'Accuracy / Precision': np.nan,
        'F1 / Recall': np.nan,
        'Пояснение': 'Сравнение fill-mask до и после доменной адаптации на текстах отзывов',
    },
    {
        'Подход': 'NER на CoNLL-2003',
        'Accuracy / Precision': ner_eval['eval_precision'],
        'F1 / Recall': ner_eval['eval_f1'],
        'Пояснение': 'Token classification с выравниванием меток и seqeval',
    },
]

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:

cls_compare_df = pd.DataFrame([
    baseline_result,
    frozen_all_result,
    freeze_5_result,
])[['experiment', 'trainable_params_after', 'accuracy', 'f1', 'runtime_sec']].sort_values('f1', ascending=False)

best_quality = cls_compare_df.iloc[0]['experiment']
best_resource = cls_compare_df.sort_values(['trainable_params_after', 'runtime_sec']).iloc[0]['experiment']

print('Итоговые выводы:')
print(f'1. Среди рассмотренных режимов классификации лучшим по качеству оказался режим: {best_quality}.')
print(f'2. Лучший по ресурсоёмкости режим: {best_resource}, так как он требует меньше обучаемых параметров и обычно быстрее обучается.')
print('3. Частичная заморозка слоёв даёт разумный компромисс между качеством и стоимостью обучения, потому что верхние слои всё ещё адаптируются под задачу.')
print('4. Few-shot SetFit особенно полезен, когда размеченных данных мало и нужно быстро получить рабочее решение без полного fine-tuning большой модели.')
print('5. Если SetFit уступает baseline, это нормально: baseline использует весь train и глубже подстраивает веса под конкретную задачу.')
print('6. Дообучение MLM полезно как доменная адаптация: модель начинает лучше отражать лексику и шаблоны конкретного корпуса текстов.')
print('7. Качественное сравнение fill-mask до и после MLM помогает увидеть, что предсказания становятся ближе к домену киноотзывов.')
print('8. В NER критически важно выравнивать метки после subword-токенизации, иначе функция потерь будет считаться по неверным позициям.')
print('9. Использование -100 для лишних под-токенов и спецтокенов делает обучение токен-классификатора корректным.')
print('10. В практической работе выбор режима зависит от ограничения по ресурсам: максимум качества даёт полный fine-tuning, а при дефиците данных и времени полезны partial freeze и SetFit.')
